# 09 — Target load and saturation

E-Perf-1 is target-load evidence; E-Perf-10 is saturation evidence. Historical eKuiper QoS-0 tails are invalid comparator evidence and are not subtracted. N, units, `thesis_evidence=false`, and descriptive-only uncertainty are explicit; missing leaves remain PENDING, never zero.


In [ ]:
import json, os
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
from wafer_analysis.focused import evidence_label, pending_record
from wafer_analysis.paths import resolve_result_batch

def passed_json(batch, artifact):
    rows = []
    for path in sorted(batch.rglob(artifact)):
        status = path.parent / 'canonical-status.json'
        if status.is_file() and json.loads(status.read_text()).get('status') == 'passed':
            rows.append((path, json.loads(path.read_text())))
    return rows

try: batch=resolve_result_batch('e-perf-10', diagnostic_path=os.environ.get('E_PERF_10_DIR'))
except (FileNotFoundError, RuntimeError, ValueError): batch=None
rows=[] if batch is None else [value for _,value in passed_json(batch,'rate-sweep.json')]
if rows:
    df=pd.DataFrame([{'system':r['system'],'offered_rate_msg_s':r['offered_rate_msg_s'],'achieved_rate_msg_s':r['achieved_rate_msg_s'],'p95_ms':r['latency_ns']['p95']/1e6,'p99_ms':r['latency_ns']['p99']/1e6,'loss_percent':r['loss_percent']} for r in rows])
    print(evidence_label(len(df), 'messages/second, milliseconds, percent', False)); display(df)
    for system, group in df.groupby('system'):
        plt.plot(group['offered_rate_msg_s'],group['p99_ms'],marker='o',label=system)
    plt.xlabel('Offered rate (messages/second)'); plt.ylabel('p99 latency (ms)'); plt.legend(); plt.title('Focused saturation diagnostic')
else:
    display(pd.DataFrame([pending_record('target load versus saturation and corrected eKuiper tail','no passed rate-sweep.json leaf','messages/second and milliseconds')]))
